# Proyecto: Algoritmo aplicado a Water Sort Puzzle

## Introducción

Nosotros seleccionamos el problema **Water Sort Game**, pues representa un desafío de optimización y combinatoria. Su estructura de juego basada en restricciones como colores, capacidades de los tubos y el cambio del liquido de tubo en tubo lo convierte en un problema muy interesante para evaluar cómo encontrar la secuencia óptima de movimientos sin caer en ciclos o explorar caminos redundantes (problemas de frontera).

Para resolverlo, vamos a implementar el algoritmo de búsqueda óptima **A* (A-Star)**. Elegimos este algoritmo porque, a diferencia de los métodos de búsqueda como BFS o DFS, A* combina el costo real acumulado con una función heurística informada ($f(n) = g(n) + h(n)$). Esto nos garantiza encontrar la solución óptima (la menor cantidad de movimientos para resolver el nivel) minimizando drásticamente el tiempo y el uso de memoria.

---

## Fases

El desarrollo se encuentra dividido en las siguientes cinco fases consecutivas:

* **Fase 1: Espacio de Estados**  
  Representar la estructura de los tubos, la distribución de los colores y las capacidades máximas de manera que Python y el algoritmo de búsqueda puedan interpretar y manipular cada configuración del juego de forma eficiente.

* **Fase 2: Reglas del Juego**  
  Definir restricciones legales para mover agua de un tubo a otro. A partir de estas reglas, se generan los estados siguientes válidos para continuar la exploración.

* **Fase 3: Diseño de la Heurística y Búsqueda**  
  Crear una función heurística $h(n)$ inteligente que estime la distancia hacia la solución. Esta función guiará al algoritmo A* para resolver el juego de manera rápida y óptima.

__Simultaneo al desarrollo del proyecto se hara uso de herramientas como canva para la creacion de la presentacion formal del proyecto.__


In [ ]:
# Importar herramientas (bibliotecas)
import heapq
import itertools

In [ ]:
# 1. CLASES BASE

class Nodo:
    def __init__(self, estado, padre=None, accion=None, costo=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.costo = costo

    def __str__(self):
        return f"Nodo(estado={self.estado}, accion={self.accion}, costo={self.costo})"


class Frontera:
    def __init__(self, f_eval):
        self.cola = []
        self.f_eval = f_eval
        self.contador = itertools.count()

    def add(self, node):
        heapq.heappush(self.cola, (self.f_eval(node), next(self.contador), node))

    def top(self):
        return self.cola[0][2]

    def pop(self):
        return heapq.heappop(self.cola)[2]

    def is_empty(self):
        return len(self.cola) == 0

def expand(problem, nodo):
    estado_act = nodo.estado
    nodos = [] 
    for (accion, next_state) in problem.result_actions(estado_act):
        cost = nodo.costo + problem.action_cost(estado_act, accion, next_state)
        nodos.append(Nodo(estado=next_state, padre=nodo, accion=accion, costo=cost))
    return nodos

## Modelado del Problema
Aquí definimos las reglas del universo del juego. Los estados serán tuplas inmutables representando los tubos y sus colores.

In [ ]:
# 2. DEFINICIÓN DEL PROBLEMA

class WaterSortProblem:
    def __init__(self, initial_state, tube_capacity=4):
        """
        initial_state: Tupla de tuplas representando los tubos iniciales.
        tube_capacity: Capacidad máxima de cada tubo (por defecto 4).
        """
        self.initial = initial_state
        self.capacity = tube_capacity

    def is_goal(self, state):
        """
        Evalúa si el tablero actual es un estado ganador.
        Victoria = Todos los tubos están vacíos, o llenos con 4 bloques del mismo color.
        """
        pass

    def result_actions(self, state):
        """
        A partir de un estado, genera todos los movimientos legales.
        Retorna una lista de tuplas: [(accion, nuevo_estado), ...]
        Una 'accion' puede ser algo como (tubo_origen, tubo_destino).
        """
        pass

    def action_cost(self, state, action, next_state):
        """
        El costo de hacer un movimiento.
        En Water Sort, cada movimiento cuesta exactamente lo mismo (1).
        """
        pass

    def heuristic(self, state):
        """
        Función h(n): Estima cuántos movimientos faltan para ganar.
        Debe ser una heurística admisible (no sobreestimar).
        """
        pass

## Algoritmo de Búsqueda: A*
Implementación del algoritmo de búsqueda del mejor primero configurado para funcionar como A*.

In [ ]:
# 3. ALGORITMO A* Y RECONSTRUCCIÓN DE RUTA

def a_star_search(problem):
    """
    Ejecuta el algoritmo Best-First Search con la función f(n) = g(n) + h(n).
    """
    def f_eval(nodo):
        return nodo.costo + problem.heuristic(nodo.estado)
    FAIL = Nodo(estado="FAIL")
    node = Nodo(estado=problem.initial, padre=None, accion=None, costo=0)
    frontera = Frontera(f_eval)
    frontera.add(node)
    reached = {node.estado: node}
    while not frontera.is_empty():
        nodo_actual = frontera.pop()
        if problem.is_goal(nodo_actual.estado):
            return nodo_actual
        expansion = expand(problem, nodo_actual)
        for hijo in expansion:
            est = hijo.estado
            if (est not in reached) or (hijo.costo < reached[est].costo):
                reached[est] = hijo
                frontera.add(hijo)        
    return FAIL

def path_actions(node):
    """
    A partir del nodo ganador, sube por los padres para extraer la secuencia de acciones.
    """
    if node.estado == "FAIL":
        return []
    acciones = []
    nodo_actual = node
    while nodo_actual is not None: 
        if nodo_actual.accion is not None:
            acciones.append(nodo_actual.accion)
        nodo_actual = nodo_actual.padre
        
    acciones.reverse()
    return acciones

## Ejecución y Pruebas
Aquí definimos un nivel de prueba, instanciamos el problema y ejecutamos el algoritmo.

In [ ]:
# 4. PRUEBA DEL ALGORITMO

nivel_inicial = (
    ("Rojo", "Azul"),
    ("Rojo", "Azul"),
    ()
)

juego = WaterSortProblem(initial_state=nivel_inicial, tube_capacity=4)

nodo_solucion = a_star_search(juego)